# Inpainting – Usuwanie Miarek z Obrazów USG
**Temat B: Miarki (caliper markers) na obrazach ultrasonograficznych**

Zestaw danych: 11 obrazów USG (21_G_R, 28_B_T, 42_SZ_K, ROK_A_J, ROK_K_M, W_BI_M)  
Miarki: **niebieskie przerywane linie pomiarowe** z endpointami '+'

---

## 1. Przegląd bibliotek do inpaintingu

| Biblioteka | Metoda | Flaga/Funkcja | Charakterystyka |
|---|---|---|---|
| **OpenCV** | Telea (FMM) | `cv2.INPAINT_TELEA` | Fast Marching, szybka, idealna dla cienkich linii |
| **OpenCV** | Navier-Stokes | `cv2.INPAINT_NS` | Przepływ płynów, lepsza dla dużych obszarów |
| **scikit-image** | Biharmonic | `inpaint_biharmonic` | Najwyższa jakość, wolna (C2-ciągłość) |
| **SimpleITK** | – | preprocessing DICOM | Brak inpaintingu, do wczytywania obrazów med. |
| **LaMa** (DNN) | FFConv | `lama-cleaner` | SOTA, wymaga GPU, omówiony w sekcji 9 |

> **Wybrana metoda główna:** Telea (r=7) — najlepszy kompromis jakość/szybkość dla wąskich linii miarek.

In [ ]:
# !pip install opencv-python-headless scikit-image matplotlib numpy

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from skimage.restoration import inpaint_biharmonic
import warnings
warnings.filterwarnings('ignore')

print('OpenCV:', cv2.__version__)

---
## 2. Wczytanie i analiza przykładowego obrazu

In [ ]:
INPUT_DIR  = Path('data')        # <-- folder z obrazami USG
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

EXAMPLE = INPUT_DIR / '21_G_R_3.jpg'
img_bgr  = cv2.imread(str(EXAMPLE))
img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

print(f'Rozmiar: {img_bgr.shape[1]}x{img_bgr.shape[0]} px')

# Analiza kanalow BGR – miarki sa niebieskie
b, g, r = img_bgr[:,:,0], img_bgr[:,:,1], img_bgr[:,:,2]
cand = ((b.astype(int)-r.astype(int) > 40) & (b > 80)).sum()
print(f'Piksele z B-R>40 i B>80 (kandydaci na miarke): {cand}')

# Typowe wartosci pikselowe miarek (z analizy rzeczywistych obrazow)
print('Typowe wartosci miarki: B~200-255, G~160-175, R~125-155')
print('Roznica B-R ~ 60-130, B-G ~ 30-90')

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, im, t in zip(axes,
    [img_rgb, b, g, r],
    ['RGB (oryginal)', 'Kanal B (niebieski)', 'Kanal G (zielony)', 'Kanal R (czerwony)']):
    ax.imshow(im, cmap='gray' if t!='RGB (oryginal)' else None)
    ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

---
## 3. Segmentacja miarek

**Kluczowa obserwacja z analizy pikseli (11 obrazow):**
Miarki USG maja wyrazna dominacje kanalu B w przestrzeni BGR:
- Centrum miarki: B≈200-255, G≈160-175, R≈125-155 
- Roznica B−R ≈ 60-130 px, B−G ≈ 30-90 px

**Strategia segmentacji:**
1. **Maska koloru:** `B − R > 30` AND `B − G > 20` AND `B > 80`  
   → selektywnie wyodrębnia niebieskie piksele miarek
2. **Dylatacja morfologiczna** (jadro eliptyczne 7×7, 2 iteracje)  
   → laczy przerwy w linii przerywanej i pokrywa grubosc linii

In [ ]:
# ── PARAMETRY SEGMENTACJI – skalibrowane na 11 obrazach USG ──────────────
SEG_PARAMS = dict(
    b_minus_r_thresh = 30,   # B - R > 30
    b_minus_g_thresh = 20,   # B - G > 20
    b_min_val        = 80,   # eliminuje ciemne obszary
    dilate_k         = 7,    # jadro dylatacji [px]
    dilate_iter      = 2,    # iteracje dylatacji
)

def segment_calipers(img_bgr, p):
    """Segmentuje niebieskie miarki USG na podstawie dominacji kanalu B."""
    b = img_bgr[:,:,0].astype(np.int16)
    g = img_bgr[:,:,1].astype(np.int16)
    r = img_bgr[:,:,2].astype(np.int16)
    mask = (
        (b - r > p['b_minus_r_thresh']) &
        (b - g > p['b_minus_g_thresh']) &
        (img_bgr[:,:,0] > p['b_min_val'])
    ).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, (p['dilate_k'], p['dilate_k']))
    return cv2.dilate(mask, kernel, iterations=p['dilate_iter'])


mask = segment_calipers(img_bgr, SEG_PARAMS)
overlay = img_rgb.copy()
overlay[mask > 0] = [255, 50, 50]
coverage = 100 * float(mask.sum()) / (255.0 * mask.size)
print(f'Pokrycie maski: {coverage:.3f}%')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, im, t in zip(axes,
    [img_rgb, mask, overlay],
    ['Oryginal', 'Maska segmentacji', 'Maska nalozona (czerwona)']):
    ax.imshow(im, cmap='gray' if t == 'Maska segmentacji' else None)
    ax.set_title(t); ax.axis('off')
plt.tight_layout(); plt.show()

---
## 4. Inpainting – porownanie metod

In [ ]:
INPAINT_RADIUS = 7

# Telea
result_telea = cv2.inpaint(img_bgr, mask, INPAINT_RADIUS, cv2.INPAINT_TELEA)
# Navier-Stokes
result_ns    = cv2.inpaint(img_bgr, mask, INPAINT_RADIUS, cv2.INPAINT_NS)
# Biharmonic (scikit-image) – tylko szarosc, wolna
gray_f = img_gray.astype(float) / 255.0
result_bih_gray = inpaint_biharmonic(gray_f, mask.astype(bool))
result_bih_u8 = (result_bih_gray * 255).clip(0,255).astype(np.uint8)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, im, t in zip(axes.flat,
    [img_rgb,
     cv2.cvtColor(result_telea, cv2.COLOR_BGR2RGB),
     cv2.cvtColor(result_ns,    cv2.COLOR_BGR2RGB),
     cv2.cvtColor(result_bih_u8, cv2.COLOR_GRAY2RGB)],
    ['Oryginal (z miarkami)',
     f'Telea (r={INPAINT_RADIUS})',
     f'Navier-Stokes (r={INPAINT_RADIUS})',
     'Biharmonic (skimage, szarosc)']):
    ax.imshow(im); ax.set_title(t, fontsize=12); ax.axis('off')
plt.suptitle('Porownanie metod inpaintingu – 21_G_R_3.jpg', fontsize=13)
plt.tight_layout(); plt.show()

---
## 5. Przetwarzanie wsadowe – wszystkie 11 obrazow

In [ ]:
image_paths = sorted(INPUT_DIR.glob('*.jpg'))
print(f'Znaleziono {len(image_paths)} obrazow')

all_results = []
for fp in image_paths:
    img = cv2.imread(str(fp))
    msk = segment_calipers(img, SEG_PARAMS)
    restored = cv2.inpaint(img, msk, INPAINT_RADIUS, cv2.INPAINT_TELEA)
    cv2.imwrite(str(OUTPUT_DIR / f'{fp.stem}_telea.jpg'), restored)
    cv2.imwrite(str(OUTPUT_DIR / f'{fp.stem}_mask.png'),  msk)
    cov = 100 * float(msk.sum()) / (255.0 * msk.size)
    all_results.append({'name': fp.name, 'coverage': cov,
                         'img': img, 'mask': msk, 'restored': restored})
    print(f'  {fp.name:28s}  maska={cov:.3f}%')

print(f'Zapisano {len(all_results)} obrazow do {OUTPUT_DIR}')

In [ ]:
# Siatka wszystkich wynikow – 2 kolumny: oryginal | inpainting
n = len(all_results)
fig, axes = plt.subplots(n, 3, figsize=(15, n*3))
for i, r in enumerate(all_results):
    overlay2 = cv2.cvtColor(r['img'], cv2.COLOR_BGR2RGB).copy()
    overlay2[r['mask']>0] = [255,50,50]
    axes[i,0].imshow(cv2.cvtColor(r['img'],     cv2.COLOR_BGR2RGB))
    axes[i,0].set_title(r['name'], fontsize=7); axes[i,0].axis('off')
    axes[i,1].imshow(overlay2)
    axes[i,1].set_title(f"Maska ({r['coverage']:.2f}%)", fontsize=7); axes[i,1].axis('off')
    axes[i,2].imshow(cv2.cvtColor(r['restored'],cv2.COLOR_BGR2RGB))
    axes[i,2].set_title('Telea r=7', fontsize=7); axes[i,2].axis('off')
plt.suptitle('Wszystkie obrazy: oryginal | maska | inpainting', fontsize=13, y=1.001)
plt.tight_layout(); plt.show()

---
## 6. Metryki jakosci (bezreferencyjna)

Brak obrazow referencyjnych → uzywamy metryk bezreferencyjna:
- **Wariancja Laplasjanu** – miarki dodaja falszywe krawedzie → **po inpaintingu powinna miec**
- **GradRMS** – RMS gradientu Sobela → **po inpaintingu powinien malec**

Wyniki eksperymentalne (11 obrazow, r=7, Telea):

| Plik | Maska% | LapVar oryg | LapVar inp | DeltaLapVar |
|---|---|---|---|---|
| 21_G_R_3 | 1.512 | 255 | 145 | **-109.8** |
| 28_B_T_2 | 1.604 | 271 | 156 | **-114.8** |
| 28_B_T_6 | 1.585 | 258 | 143 | **-115.3** |
| 42_SZ_K_2 | 1.433 | 259 | 153 | **-105.8** |
| ROK_A_J_1 | 1.736 | 278 | 155 | **-123.0** |
| ROK_A_J_2 | 1.720 | 277 | 157 | **-119.4** |
| ROK_A_J_3 | 1.704 | 266 | 157 | **-108.3** |
| ROK_A_J_4 | 1.711 | 267 | 153 | **-113.9** |
| ROK_K_M_1 | 1.689 | 288 | 161 | **-126.6** |
| ROK_K_M_5 | 1.710 | 314 | 204 | **-109.9** |
| W_BI_M_6 | 1.572 | 275 | 162 | **-113.7** |

We wszystkich 11 przypadkach obie metryki maleja – inpainting skutecznie usuwa artefakty.

In [ ]:
def compute_metrics(orig_bgr, rest_bgr):
    def lap_var(img):
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(float)
        return float(cv2.Laplacian(g, cv2.CV_64F).var())
    def grad_rms(img):
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(float)
        return float(np.sqrt(
            cv2.Sobel(g,cv2.CV_64F,1,0)**2 +
            cv2.Sobel(g,cv2.CV_64F,0,1)**2).mean())
    return {'LapVar_orig': lap_var(orig_bgr), 'LapVar_inp': lap_var(rest_bgr),
            'GradRMS_orig': grad_rms(orig_bgr), 'GradRMS_inp': grad_rms(rest_bgr)}

metrics_all = []
for r in all_results:
    m = compute_metrics(r['img'], r['restored'])
    m['name'] = r['name']
    metrics_all.append(m)
    print(f"{r['name']:26s}  "
          f"LapVar: {m['LapVar_orig']:5.0f} -> {m['LapVar_inp']:5.0f}  "
          f"(delta={m['LapVar_inp']-m['LapVar_orig']:+.0f})  "
          f"GradRMS: {m['GradRMS_orig']:5.2f} -> {m['GradRMS_inp']:5.2f}")

In [ ]:
names_s = [m['name'].replace('.jpg','') for m in metrics_all]
x = np.arange(len(names_s)); w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.bar(x-w/2, [m['LapVar_orig'] for m in metrics_all], w, label='Oryginal', color='#e74c3c', alpha=0.85)
ax.bar(x+w/2, [m['LapVar_inp']  for m in metrics_all], w, label='Inpainting', color='#2ecc71', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(names_s, rotation=35, ha='right', fontsize=8)
ax.set_ylabel('Wariancja Laplasjanu'); ax.set_title('LapVar: oryginal vs inpainting')
ax.legend(); ax.grid(axis='y', alpha=0.4)

ax2 = axes[1]
ax2.bar(x-w/2, [m['GradRMS_orig'] for m in metrics_all], w, label='Oryginal', color='#3498db', alpha=0.85)
ax2.bar(x+w/2, [m['GradRMS_inp']  for m in metrics_all], w, label='Inpainting', color='#f39c12', alpha=0.85)
ax2.set_xticks(x); ax2.set_xticklabels(names_s, rotation=35, ha='right', fontsize=8)
ax2.set_ylabel('GradRMS'); ax2.set_title('GradRMS: oryginal vs inpainting')
ax2.legend(); ax2.grid(axis='y', alpha=0.4)

plt.suptitle('Metryki bezreferencyjna – 11 obrazow USG', fontsize=13)
plt.tight_layout(); plt.show()

---
## 7. Dobor optymalnego promienia

In [ ]:
img_test  = cv2.imread(str(INPUT_DIR / 'ROK_A_J_1.jpg'))
mask_test = segment_calipers(img_test, SEG_PARAMS)

radii = [3, 5, 7, 10, 15, 20]
lap_scores = []

fig, axes = plt.subplots(1, len(radii), figsize=(20, 4))
for ax, rad in zip(axes, radii):
    rest = cv2.inpaint(img_test, mask_test, rad, cv2.INPAINT_TELEA)
    g_rest = cv2.cvtColor(rest, cv2.COLOR_BGR2GRAY).astype(float)
    lv = float(cv2.Laplacian(g_rest, cv2.CV_64F).var())
    lap_scores.append(lv)
    ax.imshow(cv2.cvtColor(rest, cv2.COLOR_BGR2RGB))
    ax.set_title(f'r={rad}\nLapVar={lv:.0f}', fontsize=9); ax.axis('off')
plt.suptitle('Tuning promienia – ROK_A_J_1.jpg', fontsize=11)
plt.tight_layout(); plt.show()

# Wykres zależnosci
plt.figure(figsize=(7, 4))
plt.plot(radii, lap_scores, 'o-', color='steelblue', lw=2, ms=8)
best_r = radii[int(np.argmin(lap_scores))]
plt.axvline(best_r, color='red', ls='--', label=f'opt r={best_r}')
plt.xlabel('Promien r [px]'); plt.ylabel('LapVar')
plt.title('Dobor promienia inpaintingu (Telea)')
plt.legend(); plt.grid(alpha=0.4); plt.tight_layout(); plt.show()
print(f'Optymalny promien: r={best_r}  LapVar={min(lap_scores):.1f}')

# Eksperymentalne wyniki:
# r:     3      5      7     10     15     20
# LapVar: 155.5  155.3  155.3  155.4  155.5  155.6
# Wniosek: r=5-7 minimalizuje LapVar; wybrano r=7 (wystarczajacy zapas na szersze linie)

---
## 8. Plansza wynikowa – reprezentatywne przypadki

In [ ]:
selected = ['21_G_R_3.jpg', 'ROK_A_J_1.jpg', 'ROK_K_M_1.jpg', 'W_BI_M_6.jpg']
fig, axes = plt.subplots(len(selected), 3, figsize=(15, 4*len(selected)))

for row, name in enumerate(selected):
    img = cv2.imread(str(INPUT_DIR / name))
    msk = segment_calipers(img, SEG_PARAMS)
    restored = cv2.inpaint(img, msk, 7, cv2.INPAINT_TELEA)
    overlay2 = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).copy()
    overlay2[msk > 0] = [255, 50, 50]
    axes[row,0].imshow(cv2.cvtColor(img,     cv2.COLOR_BGR2RGB))
    axes[row,0].set_title(f'{name} – Oryginal', fontsize=9); axes[row,0].axis('off')
    axes[row,1].imshow(overlay2)
    axes[row,1].set_title('Maska segmentacji', fontsize=9); axes[row,1].axis('off')
    axes[row,2].imshow(cv2.cvtColor(restored,cv2.COLOR_BGR2RGB))
    axes[row,2].set_title('Inpainting Telea r=7', fontsize=9); axes[row,2].axis('off')

plt.suptitle('Wyniki inpaintingu miarek USG', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()

---
## 9. Zadanie dodatkowe – DNN do inpaintingu

| Model | Rok | Kluczowy pomysl | Ocena dla miarek USG |
|---|---|---|---|
| Context Encoder | 2016 | Enc-dec + GAN | Podstawowy; wymaga treningu |
| DeepFill v2 | 2019 | Gated conv + attention | Dobry dla duzych masek |
| **LaMa** | 2022 | Fast Fourier Convolutions | **Najlepszy kandydat** dla regularnych wzorcow |
| MAT | 2022 | Multi-scale Transformer | SOTA jakos, wolny bez GPU |
| SD Inpainting | 2022+ | Dyfuzja latentna | Ryzyko halucynacji anatomicznych |

**Wniosek:** LaMa bylby optymalny – FFC zapewnia globalny kontekst niezbedny do
uzupelniania dlugich linii miarek. Jednak dla zastosowan diagnostycznych konieczna
jest walidacja kliniczna: modele DNN moga generowac struktury anatomiczne  
ktorych nie ma, nieodroznialne wizualnie od prawdziwej tkanki.

---
## Podsumowanie

| Aspekt | Wynik |
|---|---|
| Metoda segmentacji | Maska koloru (B-R>30, B-G>20, B>80) + dylatacja 7px×2 |
| Pokrycie maski | ~1.4–1.7% pikseli (spojne we wszystkich 11 obrazach) |
| Metoda inpaintingu | Telea (FMM), r=7 |
| Metryka LapVar | DeltaLapVar = -105 do -127 (poprawa we wszystkich obrazach) |
| Czas przetwarzania | ~0.1 s / obraz (CPU) |